```
Preprocesamiento.py  ·  Reto 9  ·  Parte 1: datos, cartera, series y volatilidad
DATASET DE ENTRADA : datos/originales/*_marketstack.csv   (26 CSV del Dow Jones, dados por el reto)
COMPLEMENTO        : Yahoo Finance (yfinance). Se descarga en memoria en cada ejecución:
                     NO se crea ninguna carpeta ni caché de Yahoo.
SALIDAS            : datos/transformados/  datos_unidos.csv · diario_marketstack.csv · precios_diarios.csv
                                           retornos_log.csv · estadisticos_descriptivos.csv
                                           parametros_simulacion.json   (σ, ρ, μ, S0... para Monte Carlo)
                     figuras/*.png   ·   graficos_js/datos_cartera.json
EJECUCIÓN          : desde la raíz del repo. Cada bloque "# %%" es una celda (VS Code / Jupyter / Spyder).
LIBRERÍAS          : pip install pandas numpy scipy matplotlib seaborn statsmodels yfinance
```

# 1. Configuración

In [ ]:
import os, json, time, warnings
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.dates, matplotlib.ticker
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats, optimize
from scipy.signal import lfilter
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.graphics.tsaplots import plot_acf

try:
    import yfinance as yf
except ImportError:          # sin yfinance el script sigue solo con los datos del reto
    yf = None
warnings.filterwarnings("ignore")

In [ ]:
# ── Carpetas ────────────────────────────────────────────────────────────────
if not os.path.isdir("datos") and os.path.isdir("../datos"):
    os.chdir("..")
RUTA_ORIG, RUTA_TRANSF = "datos/originales", "datos/transformados"
RUTA_FIG, RUTA_JS = "figuras", "graficos_js"
for carpeta in (RUTA_TRANSF, RUTA_FIG, RUTA_JS):          # (no se crea ninguna carpeta de Yahoo)
    os.makedirs(carpeta, exist_ok=True)

# ── Parámetros del reto (Descripción técnica §4-§5) ─────────────────────────
TTM, N_PASOS, N_SIMS = 5, 20, 100_000      # años · instantes · simulaciones
MU_RF, E_STRIKE = 0.01, 3.0                # tasa libre de riesgo (= μ de los 3 activos) · strike
DIAS_ANIO = 252                            # días de cotización al año (para anualizar)

# ── Parámetros del análisis ─────────────────────────────────────────────────
FIN_ANALISIS = pd.Timestamp("2021-06-30")  # último día completo del dataset
INICIO_IDEAL = pd.Timestamp("2018-06-06")  # primer día del dataset (Yahoo permite dar esa historia a los 26)
TOL = 0.015                                # si marketstack y Yahoo difieren más de un 1,5 % → dato anómalo
CARTERA_MANUAL = None                      # p. ej. ["CRM","JPM","PG"] para fijar la cartera; None = automática

# ── Estilo de las figuras (mismos colores en todo el script) ────────────────
TEAL, DARK, ACCENT, GRIS = "#5FA3AD", "#1F4B52", "#E07A3F", "#B8BEC2"
COL3 = [DARK, TEAL, ACCENT]                # un color fijo por activo de la cartera
sns.set_theme(style="whitegrid", rc={"axes.edgecolor": GRIS, "grid.color": "#E6E9EB"})
mpl.rcParams.update({"figure.dpi": 110, "axes.titleweight": "bold", "axes.titlesize": 12})
pct = mpl.ticker.PercentFormatter(1.0, decimals=0)

def guardar(fig, nombre):
    fig.savefig(f"{RUTA_FIG}/{nombre}.png", dpi=200, bbox_inches="tight")

In [ ]:
# ── Información cualitativa de las 26 empresas (sector GICS en 2021) ────────
SECTOR = {
 "AAPL":"Tecnología","MSFT":"Tecnología","CSCO":"Tecnología","IBM":"Tecnología","INTC":"Tecnología","CRM":"Tecnología","V":"Tecnología",
 "AMGN":"Salud","JNJ":"Salud","MRK":"Salud","UNH":"Salud",
 "AXP":"Financiero","GS":"Financiero","JPM":"Financiero","TRV":"Financiero",
 "BA":"Industria","CAT":"Industria","MMM":"Industria","CVX":"Energía","DIS":"Comunicación","DOW":"Materiales",
 "HD":"Consumo discrecional","MCD":"Consumo discrecional","NKE":"Consumo discrecional","KO":"Consumo básico","PG":"Consumo básico"}
NOMBRE = {"AAPL":"Apple","AMGN":"Amgen","AXP":"American Express","BA":"Boeing","CAT":"Caterpillar","CRM":"Salesforce","CSCO":"Cisco",
 "CVX":"Chevron","DIS":"Disney","DOW":"Dow Inc.","GS":"Goldman Sachs","HD":"Home Depot","IBM":"IBM","INTC":"Intel","JNJ":"Johnson & Johnson",
 "JPM":"JPMorgan","KO":"Coca-Cola","MCD":"McDonald's","MMM":"3M","MRK":"Merck","MSFT":"Microsoft","NKE":"Nike","PG":"Procter & Gamble",
 "TRV":"Travelers","UNH":"UnitedHealth","V":"Visa"}
# Según los apuntes: sectores cíclicos (tecnología, turismo, banca...) más volátiles; defensivos (consumo básico, salud) más estables.
DEFENSIVOS = {"Consumo básico", "Salud"}
PAL_SECTOR = dict(zip(sorted(set(SECTOR.values())), sns.color_palette("tab10", 9)))

# Festivos NYSE (jun-2018 → jun-2021): sirven para saber qué sesiones debería haber y detectar huecos.
FESTIVOS = pd.to_datetime(["2018-07-04","2018-09-03","2018-11-22","2018-12-05","2018-12-25","2019-01-01","2019-01-21","2019-02-18",
 "2019-04-19","2019-05-27","2019-07-04","2019-09-02","2019-11-28","2019-12-25","2020-01-01","2020-01-20","2020-02-17","2020-04-10",
 "2020-05-25","2020-07-03","2020-09-07","2020-11-26","2020-12-25","2021-01-01","2021-01-18","2021-02-15","2021-04-02","2021-05-31"])
def dias_de_mercado(ini, fin):
    d = pd.bdate_range(ini, fin)
    return d[~d.isin(FESTIVOS)]

# 2. Lectura y unificación de los 26 CSV
Se mantiene la función `unir_csvs` del equipo (solo se añade la creación de la carpeta de salida).

In [ ]:
def unir_csvs(carpeta="datos/originales", salida="datos/transformados/datos_unidos.csv"):
    dfs = []

    for archivo in os.listdir(carpeta):
        if archivo.endswith(".csv"):
            ruta = os.path.join(carpeta, archivo)
            df = pd.read_csv(ruta)
            dfs.append(df)

    df_final = pd.concat(dfs, ignore_index=True)

    os.makedirs(os.path.dirname(salida), exist_ok=True)
    df_final.to_csv(salida, index=False)

    return df_final

df = unir_csvs(RUTA_ORIG, f"{RUTA_TRANSF}/datos_unidos.csv")
df["date"] = pd.to_datetime(df["date"], utc=True)
print(f"{len(df):,} filas · {df['symbol'].nunique()} símbolos · {df['date'].min()} → {df['date'].max()}")

# 3. Diagnóstico de calidad
Según los apuntes, los datos de mercado son *ticks* que se resumen en formato **OHLC** (+ `last` y `volume`).
Aquí hay que comprobar qué contiene realmente cada fila antes de usarla.

In [ ]:
print("% de nulos por columna:")
print((df[["open","high","low","last","close","volume"]].isna().mean() * 100).round(1).to_string())
print("\nexchange:", df["exchange"].unique().tolist(), "(constante, no aporta)")
print("Duplicados (symbol, date):", df.duplicated(subset=["symbol", "date"]).sum())
print("high < low:", (df["high"] < df["low"]).sum(), "· precios <= 0:", (df[["open","high","low","close"]] <= 0).any(axis=1).sum())

resumen = df.groupby("symbol").agg(filas=("date", "size"), desde=("date", "min"), hasta=("date", "max"))
resumen["desde"], resumen["hasta"] = resumen["desde"].dt.date, resumen["hasta"].dt.date
print(resumen.to_string())

# Cobertura: registros por símbolo y mes
cob = df.assign(mes=df["date"].dt.strftime("%Y-%m")).pivot_table(index="symbol", columns="mes", values="open", aggfunc="size", fill_value=0)
fig, ax = plt.subplots(figsize=(13, 6))
sns.heatmap(np.log10(cob.replace(0, np.nan)), cmap=sns.light_palette(DARK, as_cmap=True), cbar_kws={"label": "log10(nº registros)"}, ax=ax)
ax.set_title("Registros por símbolo y mes: 6 tickers desde 2018, 20 desde mayo-2020"); ax.set_xlabel(""); ax.set_ylabel("")
plt.setp(ax.get_xticklabels(), rotation=90, fontsize=7); plt.tight_layout(); guardar(fig, "01_cobertura"); plt.show()

**Hallazgo:** el fichero mezcla **dos formatos** bajo las mismas columnas.

* **Régimen A (antes de mayo-2020, solo AAPL, CAT, GS, INTC, NKE, V):** barras OHLC de 5-30 min; `last` y `volume` vacíos; las horas
  vienen desplazadas respecto a la sesión NYSE. La última barra del día es la subasta de cierre (O=H=L=C).
* **Régimen B (desde mayo-2020):** snapshots casi por minuto. `open/high/low` son **acumulados del día**, `last` es el precio actual,
  `volume` es el volumen **acumulado del día solo en IEX** (una pequeña parte del real) y **`close` es el cierre del día anterior**.

Por tanto no se puede usar `close` tal cual ni promediar filas: hay que **reconstruir una fila por sesión**.

In [ ]:
# Evidencia: régimen B (AAPL 28-jun-2021) y régimen A (AAPL 6-jun-2018)
b = df[(df.symbol == "AAPL") & df.date.between("2021-06-28 13:30", "2021-06-28 20:00") & df["last"].notna()].sort_values("date")
print(b[["date", "open", "high", "low", "last", "close", "volume"]].iloc[::70].to_string())
a = df[(df.symbol == "AAPL") & df.date.between("2018-06-06 17:00", "2018-06-07 01:00")].sort_values("date")
print(a[["date", "open", "high", "low", "last", "close", "volume"]].iloc[[0, 1, 2, -3, -2, -1]].to_string())

# 4. Limpieza: de ticks a una serie diaria
1. **Régimen A:** una sesión = bloque de barras sin huecos de más de 6 h. Cierre = `close` de la última barra.
2. **Régimen B:** solo filas con `last`, entre 09:30 y 16:00 (hora de Nueva York). Cierre = el `close` que aparece al día siguiente (es el cierre previo);
   si difiere más de un 1 % de `last` se usa `last`. Volumen = máximo del acumulado.
3. **Splits** (apuntes: *acciones de empresa*): el precio cae pero el valor de la empresa no. Un salto |ln retorno| > 0,6 compatible con un ratio entero
   (2:1, 4:1...) se corrige dividiendo el histórico anterior (AAPL 4:1, agosto-2020).

In [ ]:
def sesiones_A(g):
    """Barras de 5-30 min → una fila por sesión."""
    g = g.sort_values("date")
    sesion = (g["date"].diff() > pd.Timedelta("6h")).cumsum()
    filas = []
    for _, s in g.groupby(sesion):
        if len(s) >= 10:                                        # se descartan sesiones incompletas
            filas.append({"fecha": s["date"].iloc[0].normalize().tz_localize(None), "open": s["open"].iloc[0],
                          "high": s["high"].max(), "low": s["low"].min(), "close": s["close"].iloc[-1],
                          "volume": np.nan, "regimen": "A"})
    return pd.DataFrame(filas)

def sesiones_B(g):
    """Snapshots con open/high/low acumulados y close = cierre previo → una fila por sesión."""
    g = g[g["last"].notna()].sort_values("date").copy()
    g["hora_ny"] = g["date"].dt.tz_convert("America/New_York")
    minuto = g["hora_ny"].dt.hour * 60 + g["hora_ny"].dt.minute
    g = g[(minuto >= 570) & (minuto <= 960) & (g["hora_ny"].dt.dayofweek < 5)].copy()      # 09:30-16:00
    g["fecha"] = g["hora_ny"].dt.tz_localize(None).dt.normalize()
    d = g.groupby("fecha").agg(open=("open", "first"), high=("high", "max"), low=("low", "min"),
                               close_last=("last", "last"), close_previo=("close", "first"), volume=("volume", "max"))
    d["close_oficial"] = d["close_previo"].shift(-1)                  # el cierre del día D aparece como "close" en D+1
    d["dif_rel"] = (d["close_oficial"] - d["close_last"]).abs() / d["close_last"]
    d["close"] = np.where(d["dif_rel"] <= 0.01, d["close_oficial"], d["close_last"])
    d["regimen"] = "B"
    return d.reset_index().drop(columns=["close_previo", "close_oficial"])

def diario_por_simbolo(g):
    primer_b = g.loc[g["last"].notna(), "date"].min()               # instante en que empieza el régimen B
    B = sesiones_B(g)
    A = sesiones_A(g[g["date"] < primer_b])
    A = A[A["fecha"] < B["fecha"].min()] if len(A) else A
    out = pd.concat([A, B], ignore_index=True).sort_values("fecha")
    out["symbol"] = g["symbol"].iloc[0]
    return out

def ajustar_splits(g):
    g = g.sort_values("fecha").copy()
    ln_ret = np.log(g["close"]).diff()
    for i in ln_ret.index[ln_ret.abs() > 0.6]:
        ratio = g["close"].shift(1)[i] / g.loc[i, "close"]
        k = min([2, 3, 4, 5, 10, 1/2, 1/3, 1/4, 1/10], key=lambda k: abs(np.log(ratio / k)))
        if abs(np.log(ratio / k)) < 0.15:
            antes = g["fecha"] < g.loc[i, "fecha"]
            g.loc[antes, ["open", "high", "low", "close", "close_last"]] /= k
            g.loc[antes, "volume"] *= k
            print(f"  Split detectado: {g['symbol'].iloc[0]} {g.loc[i, 'fecha'].date()} ratio {k:g}:1 → histórico ajustado")
    return g

diario = pd.concat([diario_por_simbolo(g) for _, g in df.groupby("symbol")], ignore_index=True)
print(f"{len(diario):,} sesiones diarias ·", diario["regimen"].value_counts().to_dict())
dif = diario.loc[diario.regimen == "B", "dif_rel"]
print(f"Régimen B: cierre oficial vs último 'last' → diferencia mediana {dif.median():.2%}; {(dif > 0.01).mean():.1%} descartadas por >1 %")

diario = pd.concat([ajustar_splits(g) for _, g in diario.groupby("symbol")], ignore_index=True)
diario.to_csv(f"{RUTA_TRANSF}/diario_marketstack.csv", index=False)

In [ ]:
# ── Huecos: sesiones que debería haber (calendario NYSE) y no están ─────────
ms = diario.pivot(index="fecha", columns="symbol", values="close").sort_index()
TICKERS = list(ms.columns)
esperadas = dias_de_mercado("2020-06-01", FIN_ANALISIS)
n_tickers = ms.reindex(esperadas).notna().sum(axis=1)
print(f"Sesiones esperadas desde 2020-06-01: {len(esperadas)}")
print("Sin dato en ningún ticker :", n_tickers[n_tickers == 0].index.strftime("%Y-%m-%d").tolist())
print("Con dato en solo algunos  :", {k.strftime("%Y-%m-%d"): int(v) for k, v in n_tickers[(n_tickers > 0) & (n_tickers < len(TICKERS))].items()})

# 5. Completar con Yahoo Finance
Yahoo aporta lo que falta en marketstack/IEX: **días perdidos**, la **historia 2018-2020** de los 20 tickers que empiezan en mayo-2020, el **volumen
consolidado** y el **calendario oficial** (índice `^DJI`). Además sirve para **validar** el cierre día a día.

Se usa `Ticker.history()` como en clase. Con `auto_adjust=False` aparecen `Close` y `Adj Close`; desde diciembre de 2024 el valor por defecto es `True`,
así que se indica explícitamente. `actions=True` añade `Dividends` y `Stock Splits` (equivalentes a `ticker.dividends` / `ticker.splits`).
En la práctica `Close` ya viene ajustado por splits y `Adj Close` además por dividendos; el enunciado no considera dividendos, así que se usa `Close`.

**Regla de fusión:** para cada día del calendario oficial se usa marketstack (ya limpio) salvo que discrepe >1,5 % de Yahoo (→ se sustituye) o no exista (→ se rellena).
Todo se hace **en memoria**: no se guarda nada de Yahoo. Si no hay conexión el script continúa solo con marketstack.

In [ ]:
def descargar_yahoo(simbolos, inicio, fin):
    """Devuelve un DataFrame largo con el histórico diario de Yahoo, o None si no hay datos."""
    if yf is None:
        return None
    trozos = []
    for s in simbolos:
        try:
            h = yf.Ticker(s).history(start=inicio.strftime("%Y-%m-%d"), end=(fin + pd.Timedelta(days=1)).strftime("%Y-%m-%d"),
                                     auto_adjust=False, actions=True)
            h = h.reindex(columns=["Close", "Volume", "Stock Splits"]).rename(columns=str.lower)
            h.index = pd.to_datetime(h.index).tz_localize(None).normalize()
            trozos.append(h.dropna(subset=["close"]).assign(symbol=s).rename_axis("fecha").reset_index())
            time.sleep(0.3)                                       # cortesía con el servidor
        except Exception as e:
            print(f"  ⚠️ Yahoo sin datos de {s}: {str(e)[:50]}")
    return pd.concat(trozos, ignore_index=True) if trozos else None

yahoo = descargar_yahoo(TICKERS + ["^DJI"], INICIO_IDEAL - pd.Timedelta(days=10), FIN_ANALISIS)
YAHOO_OK = yahoo is not None and yahoo["symbol"].nunique() >= 20
print("Yahoo Finance:", "OK" if YAHOO_OK else "NO disponible → solo marketstack")
if YAHOO_OK:
    yf_close = yahoo.pivot(index="fecha", columns="symbol", values="close")
    yf_vol = yahoo.pivot(index="fecha", columns="symbol", values="volume")
    print("Splits según Yahoo:", yahoo.loc[yahoo["stock splits"] > 0, ["fecha", "symbol", "stock splits"]].assign(fecha=lambda x: x.fecha.dt.date).values.tolist())

In [ ]:
# ── Calendario y ventana ────────────────────────────────────────────────────
if YAHOO_OK and "^DJI" in yf_close:
    calendario, INICIO = yf_close["^DJI"].dropna().index, INICIO_IDEAL       # calendario oficial de Yahoo
else:
    todos = dias_de_mercado(ms.index.min(), FIN_ANALISIS)
    INICIO = ms.reindex(todos).notna().mean(axis=1).loc[lambda s: s >= 0.9].index.min()   # 1.er día con casi todos los tickers
    calendario = todos
calendario = calendario[(calendario >= INICIO) & (calendario <= FIN_ANALISIS)]
print(f"Ventana de análisis: {calendario.min().date()} → {calendario.max().date()} ({len(calendario)} sesiones)")

# ── Fusión marketstack + Yahoo ──────────────────────────────────────────────
precios = pd.DataFrame(index=calendario, columns=TICKERS, dtype=float)
volumen = pd.DataFrame(index=calendario, columns=TICKERS, dtype=float)
auditoria = []
for t in TICKERS:
    base = ms[t].reindex(calendario)
    n_ms, n_sust, n_rell = int(base.notna().sum()), 0, 0
    if YAHOO_OK and t in yf_close:
        y = yf_close[t].reindex(calendario)
        anomalo = ((base / y - 1).abs() > TOL) & base.notna() & y.notna()      # marketstack difiere de Yahoo
        base = base.where(~anomalo)
        n_sust, n_rell = int(anomalo.sum()), int((base.isna() & y.notna()).sum())
        base = base.combine_first(y)
        volumen[t] = yf_vol[t].reindex(calendario)                              # volumen consolidado
    else:
        volumen[t] = diario[diario.symbol == t].set_index("fecha")["volume"].reindex(calendario)   # solo IEX
    precios[t] = base
    auditoria.append({"symbol": t, "de_marketstack": n_ms - n_sust, "sustituidos": n_sust, "rellenados_yahoo": n_rell,
                      "sin_dato": int(base.isna().sum()), "cobertura": round(base.notna().mean(), 3)})
print(pd.DataFrame(auditoria).set_index("symbol").to_string())

In [ ]:
# ── Rentabilidades ──────────────────────────────────────────────────────────
# rent = rentabilidad simple diaria (como en clase: precio_t / precio_{t-1} - 1)
# ret  = rentabilidad logarítmica ln(1+rent): es la que usa el movimiento browniano geométrico del reto.
p = precios.ffill(limit=5)
rent = (p / p.shift(1) - 1)
rent[precios.isna()] = np.nan                     # un día sin precio NO es rentabilidad 0
rent = rent.iloc[1:]
ret = np.log1p(rent)

precios.to_csv(f"{RUTA_TRANSF}/precios_diarios.csv")
ret.to_csv(f"{RUTA_TRANSF}/retornos_log.csv")
print(f"precios {precios.shape} · retornos {ret.shape}")
grandes = ret.stack()[lambda s: s.abs() > 0.12]
print(f"Retornos con |r| > 12 %: {len(grandes)} (COVID mar-2020, vacuna 9-nov-2020, resultados...). Son reales: NO se eliminan.")

# 6. Análisis descriptivo del universo
Se usan las medidas vistas en clase: **rentabilidad activa**, **rentabilidad anualizada** `(1+r)^(252/n) − 1`, **volatilidad anualizada** `σ·√252`,
**Sharpe** `(r − rf)/σ` (con rf = 1 %, la del reto), **Sortino** (solo la volatilidad de las rentabilidades negativas) y **drawdown máximo**.

In [ ]:
n = precios.count()
primero = precios.apply(lambda s: s.dropna().iloc[0])
ultimo = precios.apply(lambda s: s.dropna().iloc[-1])
mercado = np.log(yf_close["^DJI"].reindex(rent.index)).diff() if YAHOO_OK and "^DJI" in yf_close else rent.mean(axis=1)

est = pd.DataFrame({"sector": pd.Series(SECTOR), "cobertura": precios.notna().mean()})
est["rent_activa"] = ultimo / primero - 1
est["rent_anual"] = (1 + est["rent_activa"]) ** (DIAS_ANIO / n) - 1
est["vol_anual"] = rent.std(ddof=0) * np.sqrt(DIAS_ANIO)
est["sharpe"] = (est["rent_anual"] - MU_RF) / est["vol_anual"]
est["sortino"] = (est["rent_anual"] - MU_RF) / (rent[rent < 0].std(ddof=0) * np.sqrt(DIAS_ANIO))
est["max_drawdown"] = (precios / precios.cummax() - 1).min()
est["beta"] = rent.apply(lambda x: x.cov(mercado) / mercado.var())
est["skew"], est["curtosis_exc"] = rent.skew(), rent.kurt()
est["liquidez_usd"] = (precios * volumen).median()
est = est.loc[TICKERS]
est.to_csv(f"{RUTA_TRANSF}/estadisticos_descriptivos.csv")
print(est.round(3).sort_values("vol_anual").to_string())
# INTERPRETACIÓN: se espera ver a los valores defensivos (consumo básico, salud) con menor volatilidad y drawdown, y a los cíclicos
# (banca, industria, tecnología) con más volatilidad y caídas más profundas en marzo de 2020, tal y como dicen los apuntes.

In [ ]:
# Fig. A: mapa riesgo / rentabilidad
fig, ax = plt.subplots(figsize=(10, 6.5))
for sec, g in est.groupby("sector"):
    ax.scatter(g["vol_anual"], g["rent_anual"], s=90, color=PAL_SECTOR[sec], edgecolor="white", label=sec, zorder=3)
for t, r in est.iterrows():
    ax.annotate(t, (r["vol_anual"], r["rent_anual"]), xytext=(4, 4), textcoords="offset points", fontsize=8)
ax.axhline(0, color=GRIS); ax.xaxis.set_major_formatter(pct); ax.yaxis.set_major_formatter(pct)
ax.set_xlabel("Volatilidad anualizada"); ax.set_ylabel("Rentabilidad anualizada"); ax.set_title("Universo Dow Jones: riesgo vs. rentabilidad")
ax.legend(title="Sector", fontsize=8, bbox_to_anchor=(1.01, 1), loc="upper left"); plt.tight_layout(); guardar(fig, "02_riesgo_rentabilidad"); plt.show()

# Fig. B: evolución base 100 y volatilidad móvil de 30 días (como en clase) de la media del universo
base100 = precios.ffill() / precios.ffill().bfill().iloc[0] * 100
vol30 = rent.mean(axis=1).rolling(30).std(ddof=0) * np.sqrt(DIAS_ANIO)
fig, (a1, a2) = plt.subplots(2, 1, figsize=(11, 7.5), sharex=True, gridspec_kw={"height_ratios": [2, 1]})
a1.plot(base100, color=GRIS, lw=0.8); a1.plot(base100.mean(axis=1), color=DARK, lw=2.4)
a1.set_ylabel("Base 100"); a1.set_title("26 acciones (gris) y su media (azul oscuro)")
a2.fill_between(vol30.index, vol30, color=TEAL, alpha=0.6); a2.yaxis.set_major_formatter(pct)
a2.set_ylabel("Vol. 30 días"); a2.set_title("Volatilidad móvil: pico del shock de marzo-2020")
plt.tight_layout(); guardar(fig, "03_evolucion_universo"); plt.show()

# Fig. C: correlaciones entre rentabilidades (nunca entre precios: arrastran tendencia)
cm = sns.clustermap(rent.corr(), cmap="vlag", center=0, vmin=-0.2, vmax=1, figsize=(9.5, 9.5), linewidths=0.3)
cm.figure.suptitle("Correlación de rentabilidades diarias (26 acciones)", y=1.01, fontweight="bold")
guardar(cm.figure, "04_correlaciones_universo"); plt.show()

# 7. Selección de la cartera de 3 activos
**Estrategia:** cartera *diversificada de riesgo moderado, con valores líquidos de sectores distintos*.

La *call* asiática se calcula sobre el promedio geométrico de S/S₀ de los tres activos, así que la prima depende de la **volatilidad** de cada uno y de la
**correlación** entre ellos. Según los apuntes, la **diversificación** aprovecha correlaciones bajas para reducir riesgo sin sacrificar rentabilidad esperada.

| Criterio | Tipo | Regla |
|---|---|---|
| Calidad de datos | cuantitativo | cobertura ≥ 95 % de sesiones (descarta DOW: cotiza desde 2019) |
| Liquidez | cuantitativo | volumen en $ ≥ mediana del universo |
| Riesgo moderado | cuantitativo | σ anual entre 15 % y 60 % |
| Diversificación | cuantitativo | mínima correlación media entre pares (con rentabilidades) |
| Eficiencia | cuantitativo | Sharpe medio como desempate (peso pequeño λ) |
| Sectores | cualitativo | 3 sectores distintos (mezcla de perfil cíclico y defensivo) |

Se evalúan todos los tríos posibles y se ordena por `score = ρ medio − λ · Sharpe medio` (menor es mejor).

In [ ]:
LAMBDA = 0.10
filtro = (est["cobertura"] >= 0.95) & est["vol_anual"].between(0.15, 0.60) & (est["liquidez_usd"] >= est["liquidez_usd"].median())
candidatos = est.index[filtro].tolist()
print(f"Candidatos: {len(candidatos)} de {len(est)} · descartados: {', '.join(est.index[~filtro])}")

C = rent[candidatos].corr()
filas = []
for trio in combinations(candidatos, 3):
    if len({SECTOR[t] for t in trio}) < 3:                       # 3 sectores distintos
        continue
    rhos = [C.loc[a, b] for a, b in combinations(trio, 2)]
    filas.append({"cartera": " · ".join(trio), "rho_medio": np.mean(rhos), "rho_max": max(rhos),
                  "sharpe_medio": est.loc[list(trio), "sharpe"].mean(), "vol_media": est.loc[list(trio), "vol_anual"].mean(),
                  "sectores": " / ".join(SECTOR[t] for t in trio)})
if not filas:
    raise ValueError("Ningún trío cumple los filtros: relaja los límites o fija CARTERA_MANUAL.")
ranking = pd.DataFrame(filas)
ranking["score"] = ranking["rho_medio"] - LAMBDA * ranking["sharpe_medio"]
ranking = ranking.sort_values("score").reset_index(drop=True)
print(f"Combinaciones válidas: {len(ranking)}"); print(ranking.head(8).round(3).to_string())

CARTERA = CARTERA_MANUAL or ranking.loc[0, "cartera"].split(" · ")
print("\nCARTERA SELECCIONADA:", CARTERA)
for t in CARTERA:
    r = est.loc[t]
    print(f"• {t} ({NOMBRE[t]}, {SECTOR[t]}, {'defensivo' if SECTOR[t] in DEFENSIVOS else 'cíclico'}): σ={r.vol_anual:.1%} · Sharpe={r.sharpe:.2f} · "
          f"Sortino={r.sortino:.2f} · drawdown máx.={r.max_drawdown:.1%} · liquidez≈{r.liquidez_usd/1e6:,.0f} M$/día")
print("Correlación media de la cartera:", round(float(np.mean([rent[a].corr(rent[b]) for a, b in combinations(CARTERA, 2)])), 3),
      "· media del universo candidato:", round(float(C.values[np.triu_indices(len(C), 1)].mean()), 3))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13.5, 5.3), gridspec_kw={"width_ratios": [1.3, 1]})
ax[0].scatter(est["vol_anual"], est["sharpe"], s=55, color=GRIS, edgecolor="white")
for t, c in zip(CARTERA, COL3):
    ax[0].scatter(est.loc[t, "vol_anual"], est.loc[t, "sharpe"], s=190, color=c, edgecolor="white", zorder=4)
    ax[0].annotate(f"{t}\n{SECTOR[t]}", (est.loc[t, "vol_anual"], est.loc[t, "sharpe"]), xytext=(8, 6), textcoords="offset points", fontsize=9, color=c, fontweight="bold")
ax[0].xaxis.set_major_formatter(pct); ax[0].set_xlabel("Volatilidad anualizada"); ax[0].set_ylabel("Sharpe"); ax[0].set_title("Los tres activos en el universo")
sns.heatmap(rent[CARTERA].corr(), annot=True, fmt=".2f", cmap=sns.light_palette(DARK, as_cmap=True), vmin=0, vmax=1, square=True, cbar=False, linewidths=2, ax=ax[1])
ax[1].set_title("Correlación de rentabilidades"); plt.tight_layout(); guardar(fig, "05_seleccion_cartera"); plt.show()

# 8. Análisis de las series de la cartera
* **STL** (descomposición en tendencia + estacionalidad + resto) del log-precio → cómo se mueve cada activo en el tiempo.
* **ADF** (H₀: raíz unitaria): el precio no debería ser estacionario y la rentabilidad sí → por eso se trabaja con rentabilidades.
* **ACF de r² y Ljung-Box**: si r² está autocorrelada hay *volatility clustering* → justifica GARCH.
* **Normalidad** (histograma, QQ-plot): el GBM del reto supone rentabilidades log-normales; se mide cuánto se aleja la realidad.

In [ ]:
R3 = ret[CARTERA].dropna()                                        # muestra común (base de σ y ρ)
P3 = precios[CARTERA].loc[R3.index.min() - pd.Timedelta(days=7):].ffill()
print(f"Muestra común: {len(R3)} rentabilidades · {R3.index.min().date()} → {R3.index.max().date()}")

fig, ax = plt.subplots(2, 1, figsize=(11, 7), sharex=True, gridspec_kw={"height_ratios": [1.6, 1]})
for t, c in zip(CARTERA, COL3):
    ax[0].plot(P3[t] / P3[t].iloc[0] * 100, color=c, lw=1.8, label=f"{t} · {SECTOR[t]}"); ax[1].plot(R3[t], color=c, lw=0.7)
ax[0].legend(); ax[0].set_ylabel("Base 100"); ax[0].set_title("Evolución de los tres activos")
ax[1].yaxis.set_major_formatter(pct); ax[1].set_ylabel("Rentabilidad log. diaria"); ax[1].set_title("Rachas de alta y baja volatilidad")
plt.tight_layout(); guardar(fig, "06_series_cartera"); plt.show()

In [ ]:
# ── STL: tendencia, estacionalidad (21 sesiones ≈ 1 mes) y residuo ──────────
fig, axes = plt.subplots(4, 3, figsize=(15, 9), sharex="col")
for j, (t, c) in enumerate(zip(CARTERA, COL3)):
    y = np.log(precios[t].dropna().loc[R3.index.min():])
    res = STL(y.values, period=21, robust=True).fit()
    comp = {"Log-precio": y.values, "Tendencia": res.trend, "Estacional": res.seasonal, "Residuo": res.resid}
    for i, (nombre, serie) in enumerate(comp.items()):
        axes[i, j].plot(y.index, serie, color=c, lw=1.1)
        if j == 0: axes[i, j].set_ylabel(nombre)
    fuerza_t = max(0, 1 - np.var(res.resid) / np.var(res.trend + res.resid))          # 1 = tendencia muy marcada
    fuerza_s = max(0, 1 - np.var(res.resid) / np.var(res.seasonal + res.resid))        # 0 = sin estacionalidad
    axes[0, j].set_title(f"{t} · fuerza tendencia {fuerza_t:.2f} · estacional {fuerza_s:.2f}")
for ax_ in axes[-1]:
    ax_.xaxis.set_major_locator(mpl.dates.YearLocator()); ax_.xaxis.set_major_formatter(mpl.dates.DateFormatter("%Y"))
plt.tight_layout(); guardar(fig, "07_stl"); plt.show()
# INTERPRETACIÓN: tendencia dominante (fuerza cercana a 1) y estacionalidad débil (cercana a 0): no hay un ciclo mensual explotable,
# el resto es ruido de corto plazo. Es coherente con un proceso de difusión (GBM).

In [ ]:
# ── Estacionariedad (ADF) y autocorrelación de r² (Ljung-Box) ───────────────
filas = []
for t in CARTERA:
    p_precio = adfuller(np.log(P3[t].dropna()), autolag="AIC")[1]
    p_ret = adfuller(R3[t], autolag="AIC")[1]
    p_lb = acorr_ljungbox(R3[t] ** 2, lags=[10], return_df=True)["lb_pvalue"].iloc[0]
    filas.append({"activo": t, "ADF_p log-precio": p_precio, "ADF_p rentabilidad": p_ret, "LjungBox_p r²": p_lb,
                  "JarqueBera_p": stats.jarque_bera(R3[t]).pvalue, "skew": R3[t].skew(), "curtosis_exc": R3[t].kurt()})
print(pd.DataFrame(filas).set_index("activo").round(4).to_string())
print("ADF p<0,05 → estacionaria · Ljung-Box p<0,05 en r² → hay clustering de volatilidad · Jarque-Bera p<0,05 → NO normal")

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for j, (t, c) in enumerate(zip(CARTERA, COL3)):
    plot_acf(R3[t] ** 2, lags=30, ax=axes[0, j], title=f"{t} · ACF de r²", color=c, vlines_kwargs={"colors": c})
    stats.probplot(R3[t], dist="norm", plot=axes[1, j])
    axes[1, j].get_lines()[0].set_color(c); axes[1, j].get_lines()[1].set_color("black"); axes[1, j].set_title(f"{t} · QQ-plot vs normal")
plt.tight_layout(); guardar(fig, "08_acf_y_normalidad"); plt.show()
# INTERPRETACIÓN: las rentabilidades reales tienen colas más pesadas que la normal (limitación del GBM) y r² está autocorrelada.

# 9. Volatilidad: modelización y predicción
La volatilidad (apuntes) es la desviación típica de las rentabilidades y se anualiza con `σ·√252`. Aquí se comparan tres estimaciones:

| Modelo | Idea |
|---|---|
| **Histórica** | desviación típica de toda la muestra |
| **EWMA** (λ = 0,94) | media móvil exponencial de r²: reacciona rápido, no revierte a la media |
| **GARCH(1,1)** | σ²ₜ = ω + α·rₜ₋₁² + β·σ²ₜ₋₁ : captura el *clustering* y la **reversión a la media** |

Predicción h días: `E[σ²(T+h)] = V_L + (α+β)^(h−1) · (σ²(T+1) − V_L)`, con `V_L = ω/(1−α−β)`.
Como la opción vence a 5 años, el σ del simulador es la raíz de la **varianza media prevista durante todo el horizonte**.
Si el ajuste GARCH sale degenerado (parámetros pegados a una cota, típico de muestras cortas) se usa la σ histórica y se avisa.

In [ ]:
def garch_fit(r):
    """GARCH(1,1) por máxima verosimilitud gaussiana. r: rentabilidades log. diarias."""
    x = (np.asarray(r) - np.mean(r)) * 100                     # en % para que el optimizador trabaje bien
    x2, v0 = x ** 2, np.mean(x ** 2)

    def varianzas(p):                                          # σ²_t = ω + α·x²_{t-1} + β·σ²_{t-1}
        z = np.r_[v0, p[0] + p[1] * x2[:-1]]
        return lfilter([1.0], [1.0, -p[2]], z)

    def menos_loglik(p):
        s2 = varianzas(p)
        return 1e10 if np.any(s2 <= 0) else 0.5 * np.sum(np.log(s2) + x2 / s2)

    res = optimize.minimize(menos_loglik, [v0 * 0.05, 0.05, 0.90], method="SLSQP",
                            bounds=[(1e-8, None), (1e-6, 0.5), (1e-6, 0.9999)],
                            constraints=[{"type": "ineq", "fun": lambda p: 0.9999 - p[1] - p[2]}])
    w, a, b = res.x
    valido = res.success and a + b < 0.995 and a < 0.45 and b > 0.05          # descarta ajustes pegados a una cota
    return {"omega": w, "alpha": a, "beta": b, "persist": a + b, "s2": varianzas(res.x), "x": x, "valido": bool(valido)}

def garch_prevision(f, h):
    """Varianza diaria prevista (en decimal²) para los próximos h días."""
    vl = f["omega"] / (1 - f["persist"])
    s2_siguiente = f["omega"] + f["alpha"] * f["x"][-1] ** 2 + f["beta"] * f["s2"][-1]
    return (vl + f["persist"] ** np.arange(h) * (s2_siguiente - vl)) / 1e4

def ewma_varianza(r, lam=0.94):
    return pd.Series(np.asarray(r) ** 2).ewm(alpha=1 - lam, adjust=False).mean().values

H = TTM * DIAS_ANIO
FITS, filas = {}, []
for t in CARTERA:
    f = garch_fit(R3[t].values); FITS[t] = f
    filas.append({"activo": t, "hist": R3[t].std(ddof=0) * np.sqrt(DIAS_ANIO), "ewma_actual": np.sqrt(DIAS_ANIO * ewma_varianza(R3[t])[-1]),
                  "garch_actual": np.sqrt(DIAS_ANIO * garch_prevision(f, 1)[0]),
                  "garch_horizonte": np.sqrt(DIAS_ANIO * garch_prevision(f, H).mean()),
                  "alpha": f["alpha"], "beta": f["beta"], "valido": f["valido"]})
tab_sigma = pd.DataFrame(filas).set_index("activo")
print(tab_sigma.round(4).to_string())

In [ ]:
# Volatilidad estimada por cada modelo (mismo color por modelo en todas las figuras)
fig, axes = plt.subplots(3, 1, figsize=(11, 8.5), sharex=True)
for ax, t in zip(axes, CARTERA):
    ax.plot(R3[t].rolling(30).std(ddof=0) * np.sqrt(DIAS_ANIO), color=GRIS, lw=1.4, label="Móvil 30 días")
    ax.plot(R3.index, np.sqrt(ewma_varianza(R3[t]) * DIAS_ANIO), color=TEAL, lw=1.3, label="EWMA 0,94")
    ax.plot(R3.index, np.sqrt(FITS[t]["s2"] / 1e4 * DIAS_ANIO), color=ACCENT, lw=1.8, label="GARCH(1,1)")
    ax.axhline(tab_sigma.loc[t, "hist"], color="black", ls=":", label="σ histórica")
    ax.set_title(f"{t} · α={FITS[t]['alpha']:.3f}  β={FITS[t]['beta']:.3f}", loc="left"); ax.yaxis.set_major_formatter(pct)
axes[0].legend(ncol=4, fontsize=8); plt.tight_layout(); guardar(fig, "09_volatilidad"); plt.show()

In [ ]:
# ── Backtest: ¿qué modelo predice mejor la varianza de los próximos 21 días? ─
def backtest(r, h=21, entrenamiento=0.6, paso=21):
    r, filas = np.asarray(r), []
    for T in range(int(len(r) * entrenamiento), len(r) - h, paso):
        pasado = r[:T]
        filas.append({"real": np.mean(r[T:T + h] ** 2), "hist": np.var(pasado), "ewma": ewma_varianza(pasado)[-1],
                      "garch": garch_prevision(garch_fit(pasado), h).mean()})
    return pd.DataFrame(filas)

BT = {t: backtest(R3[t].values) for t in CARTERA}
def errores(bt):
    vol = lambda v: np.sqrt(DIAS_ANIO * v)
    return pd.DataFrame({m: {"RMSE_vol": np.sqrt(np.mean((vol(bt[m]) - vol(bt["real"])) ** 2)),
                             "QLIKE": np.mean(np.log(bt[m]) + bt["real"] / bt[m])} for m in ["hist", "ewma", "garch"]}).T
res_bt = pd.concat({t: errores(BT[t]) for t in CARTERA}, names=["activo", "modelo"])
print(res_bt.round(4).to_string()); print("\nMedia de los tres activos (menor = mejor):"); print(res_bt.groupby("modelo").mean().round(4).to_string())

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, t in zip(axes, CARTERA):
    bt = BT[t]
    ax.plot(np.sqrt(DIAS_ANIO * bt["real"]), color="black", lw=2, marker="o", ms=4, label="Realizada 21 d")
    ax.plot(np.sqrt(DIAS_ANIO * bt["hist"]), color=GRIS, label="Histórica"); ax.plot(np.sqrt(DIAS_ANIO * bt["ewma"]), color=TEAL, label="EWMA")
    ax.plot(np.sqrt(DIAS_ANIO * bt["garch"]), color=ACCENT, ls="--", label="GARCH"); ax.set_title(f"{t} · predicción a 21 días"); ax.yaxis.set_major_formatter(pct)
axes[0].legend(fontsize=8); plt.tight_layout(); guardar(fig, "10_backtest_volatilidad"); plt.show()

In [ ]:
# ── σ final para el simulador ───────────────────────────────────────────────
METODO_SIGMA = "garch_horizonte"                 # también: "hist" o "ewma_actual"
SIGMA = {}
for t in CARTERA:
    metodo = METODO_SIGMA
    if metodo == "garch_horizonte" and not tab_sigma.loc[t, "valido"]:
        print(f"⚠️ {t}: GARCH degenerado (α={tab_sigma.loc[t,'alpha']:.2f}, β={tab_sigma.loc[t,'beta']:.2f}) → σ histórica"); metodo = "hist"
    SIGMA[t] = float(tab_sigma.loc[t, metodo])
print("σ anual del simulador:", {t: round(v, 4) for t, v in SIGMA.items()})

# 10. Correlaciones y descomposición de Cholesky
Se estima ρ (3×3) con **rentabilidades** (no precios) y la misma muestra que σ. Debe ser simétrica y definida positiva para poder factorizarla.
`numpy.linalg.cholesky` devuelve la triangular **inferior** L con ρ = L·Lᵀ. (El enunciado escribe ρ = AᵀA con A triangular inferior; son equivalentes con A = Lᵀ.)
Al simular: `Z_correlado = Z_independiente @ L.T`.

In [ ]:
rho = R3.corr().values
print(pd.DataFrame(rho, CARTERA, CARTERA).round(4).to_string())
L = np.linalg.cholesky(rho)
autovalores = np.linalg.eigvalsh(rho)
print("Autovalores:", autovalores.round(4), "→ definida positiva" if autovalores.min() > 0 else "→ ¡NO definida positiva!")
print("Error máx. |L·Lᵀ − ρ| =", np.abs(L @ L.T - rho).max())

fig, ax = plt.subplots(figsize=(11, 4))
for (a, b), c in zip(combinations(CARTERA, 2), COL3):
    ax.plot(R3[a].rolling(63).corr(R3[b]), color=c, lw=1.8, label=f"{a}–{b}"); ax.axhline(R3[a].corr(R3[b]), color=c, ls=":")
ax.set_ylim(-0.4, 1); ax.legend(ncol=3); ax.set_ylabel("Correlación móvil 63 días")
ax.set_title("La correlación cambia en el tiempo (sube en crisis); ρ = media de la muestra (punteado)")
plt.tight_layout(); guardar(fig, "11_correlacion_movil"); plt.show()

# 11. Exportar los parámetros para Monte Carlo

In [ ]:
sigma_vec = np.array([SIGMA[t] for t in CARTERA])
params = {"tickers": CARTERA, "empresas": [NOMBRE[t] for t in CARTERA], "sectores": [SECTOR[t] for t in CARTERA],
          "ventana": [str(R3.index.min().date()), str(R3.index.max().date())],
          "fuente_precios": "marketstack + Yahoo Finance" if YAHOO_OK else "marketstack",
          "S0": [1.0, 1.0, 1.0], "mu": [MU_RF] * 3, "sigma": sigma_vec.tolist(), "metodo_sigma": METODO_SIGMA,
          "rho": rho.tolist(), "cholesky_L": L.tolist(),
          "TTM": TTM, "n_pasos": N_PASOS, "dt": TTM / N_PASOS, "n_sims": N_SIMS, "E": E_STRIKE}
with open(f"{RUTA_TRANSF}/parametros_simulacion.json", "w", encoding="utf-8") as f:
    json.dump(params, f, indent=2, ensure_ascii=False)

b100 = (P3 / P3.iloc[0] * 100).round(3)                     # datos para los gráficos JS
with open(f"{RUTA_JS}/datos_cartera.json", "w", encoding="utf-8") as f:
    json.dump({"fechas": b100.index.strftime("%Y-%m-%d").tolist(), "base100": {t: b100[t].tolist() for t in CARTERA},
               "correlacion": rho.round(4).tolist(), "sigma": dict(zip(CARTERA, sigma_vec.round(4))), "colores": dict(zip(CARTERA, COL3))}, f, ensure_ascii=False)

print(pd.DataFrame({"empresa": params["empresas"], "sector": params["sectores"], "S0": 1.0, "mu": MU_RF, "sigma": sigma_vec}, index=CARTERA).round(4).to_string())
print(f"dt = {TTM / N_PASOS} años · {N_PASOS} pasos · {N_SIMS:,} simulaciones · strike {E_STRIKE}")
print("Guardado en", RUTA_TRANSF, "→ siguiente paso: Simulacion_MonteCarlo.py")